# NB11 – Technologievergleich: Batteriespeicher
### CAS Information Engineering – Scripting Project (Kür)
**Gruppe:** SC26_Gruppe_2 | **Datum:** März–Mai 2026

---
Vergleich der relevanten Batterietechnologien für stationäre Grid-Arbitrage-Anwendungen:  
**LFP · NMC · Redox-Flow (Vanadium) · NaS** — nach CAPEX, Lebensdauer, Wirkungsgrad und Segmenteignung.

---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [← NB10 Dispatch](10_Dispatch_Optimierung.ipynb) | [→ NB12 Alternative Speicher](12_Alternative_Speicher.ipynb) |
|:---|:---:|---:|


---
## 0. Setup

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')

with open('config.json') as f:
    CFG = json.load(f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
CHARTS_DIR = os.path.join('output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: config.json

_viz=CFG.get('visualisierung',{}).get('farben',{})
BG_DARK  = _viz.get('bg_dark',  '#0d1117')
BG_PANEL = _viz.get('bg_panel', '#141414')
C_PRICE  = _viz.get('c_price',  '#FFA726')
C_LOAD   = _viz.get('c_load',   '#66BB6A')
C_CHARGE = _viz.get('c_charge', '#1565C0')
C_FEED   = _viz.get('c_feed',   '#B71C1C')
SEG_COLORS = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])
SEG_COLORS = ['#42A5F5','#66BB6A','#FFA726','#EF5350']

print(f"Setup OK | Szenario={SZ_AKTIV} | Charts → {CHARTS_DIR}")

---
## 1. Technologie-Steckbriefe

Kennwerte 2024/2025 — Quellen: Bloomberg NEF, IRENA, Fraunhofer ISE.  
Alle Werte: System-CAPEX (Zellen + Wechselrichter + Installation), ohne Gebäude/Fundament.


In [ ]:
# ── Technologievergleich-Tabelle ─────────────────────────────────────────────
tech_data = {
    'Technologie'         : ['LFP (Li-Ion)', 'NMC (Li-Ion)', 'Redox-Flow (VRF)', 'NaS'],
    'CAPEX [EUR/kWh]'     : ['150–300',       '200–400',       '400–800',          '300–500'],
    'Lebensdauer [J]'     : ['15–20',          '10–15',          '20–25',            '15–20'],
    'Zyklen'              : ['3 000–6 000',    '1 500–3 000',    '> 20 000',         '4 500+'],
    'RTE [%]'             : ['92–95',          '90–93',          '70–80',            '75–85'],
    'Selbstentl./Tag'     : ['2–3 %',          '2–5 %',          '< 1 %',            '~15 %'],
    'Betriebstemperatur'  : ['-20…+60 °C',     '-20…+60 °C',     '+10…+40 °C',       '+300…+350 °C'],
    'Skalierbarkeit'      : ['Modular, gut',   'Modular, gut',   'Sehr hoch',        'Mittel'],
    'Brandsicherheit'     : ['Hoch (LFP)',      'Mittel (NMC)',   'Unkritisch',       'Betriebswärme'],
    'Typisches Segment'   : ['Privat/Gewerbe', 'EV / mobil',     'Utility/Netz',     'Utility/Wärme'],
}
df_tech = pd.DataFrame(tech_data).set_index('Technologie')
display(df_tech.T)


**Projektrelevanz:** NB04 verwendet LFP als Referenztechnologie (RTE = 92 %, CAPEX 150–400 EUR/kWh je Segment).
Diese Wahl ist korrekt: LFP dominiert den CH-Stationärspeichermarkt 2023–2026.


---
## 2. Technologieprofil-Vergleich (Radar-Chart)

Normierte Scores (1–10): höher = besser für Grid-Arbitrage.  
Scoring-Kriterien: CAPEX (invertiert), Lebensdauer, Zyklenanzahl, Wirkungsgrad, Sicherheit, Skalierbarkeit.


In [ ]:
# ── Radar-Chart ──────────────────────────────────────────────────────────────
scores = {
    'LFP'        : [8.5, 7.0, 9.5, 9.5, 9.0, 8.0],
    'NMC'        : [7.0, 5.5, 6.0, 8.5, 5.0, 7.5],
    'Redox-Flow' : [4.0, 9.0, 10.0, 6.0, 10.0, 10.0],
    'NaS'        : [6.0, 7.0, 8.0, 7.0, 4.0, 5.0],
}
labels = ['CAPEX\n(inv.)', 'Lebens-\ndauer', 'Zyklen', 'Wirkungs-\ngrad', 'Sicherheit', 'Skalierung']
N = len(labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
colors_t = ['#42A5F5','#FFA726','#66BB6A','#EF5350']

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.tick_params(colors='#bbbbbb'); ax.grid(color='#333', lw=0.7)
ax.spines['polar'].set_color('#444')

for (name, sc), col in zip(scores.items(), colors_t):
    vals = sc + [sc[0]]
    ax.plot(angles, vals, color=col, lw=2.2, label=name)
    ax.fill(angles, vals, color=col, alpha=0.10)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, color='#ccc', fontsize=9)
ax.set_ylim(0, 10)
ax.set_yticks([2,4,6,8,10])
ax.set_yticklabels(['2','4','6','8','10'], color='#555', fontsize=7)
ax.set_title('Technologieprofil — stationäre Grid-Arbitrage', color='white', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.12),
          framealpha=0.3, facecolor='#111', labelcolor='white', fontsize=9)

p = os.path.join(CHARTS_DIR, 'kuer_nb11_radar.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")

---
## 3. CAPEX-Lernkurven 2015–2024 + Projektion bis 2035

### Datenquellen

| Quelle | Inhalt | Zugang |
|--------|--------|--------|
| **Our World in Data** (BNEF/Ziegler & Trancik) | Historische Zellpreise 1991–2024, USD/kWh | Freier CSV-Download |
| **NREL ATB 2024** | Systemkosten + Projektionsszenarien 2022–2050 | Freier CSV-Download (AWS S3) |
| **BloombergNEF Battery Price Survey** | Umfassendste Datenbasis: Zell- und Systemkosten, Chemie-Aufschlüsslung, Regionen | **Bezahlschranke** |

> **Hinweis zu BNEF:** BloombergNEF ist die Referenzquelle der Branche für Batteriepreis-Daten
> (auch NREL ATB und OWID beruhen partiell darauf). Für geschäftliche Entscheidungen
> — z.B. konkrete Investitionsrechnung oder Due-Diligence — wäre ein BNEF-Abonnement
> in Erwägung zu ziehen.

**Methodik:** Exponentielle Regression auf historischen Daten (Proxy für Wright’s Law);
NREL ATB Szenarien als zusätzliche Referenzkurven.


In [ ]:
# ── CAPEX-Lernkurven: OWID + NREL ATB CSV + Regression ───────────────────────
import requests, io
from scipy.optimize import curve_fit

# ════════════════════════════════════════════════════════════════════════════
# DATENQUELLE 1: Our World in Data — historische Li-Ion Zellpreise
# Rupert Way (2026) auf Basis BNEF, Ziegler & Trancik (2021), Avicenne Energy
# Direkt-CSV: https://ourworldindata.org/grapher/price-of-lithium-ion-battery-cells.csv
# ════════════════════════════════════════════════════════════════════════════
OWID_URL = 'https://ourworldindata.org/grapher/price-of-lithium-ion-battery-cells.csv'

df_hist = None
try:
    resp = requests.get(OWID_URL, timeout=15)
    resp.raise_for_status()
    _df  = pd.read_csv(io.StringIO(resp.text))
    # Spaltenname für Preis finden (variiert je nach OWID-Version)
    col_price = next((c for c in _df.columns
                      if any(k in c.lower() for k in ['price','lithium','cost','value'])), _df.columns[-1])
    col_year  = next((c for c in _df.columns if 'year' in c.lower() or c == 'Year'), 'Year')
    df_hist = (_df.groupby(col_year)[col_price].mean()
                  .reset_index().rename(columns={col_year: 'Year', col_price: 'cell_usd_kwh'}))
    # Systemkosten: Zellkosten × 2.5 (BOS + WR + Installation) × 0.92 (USD→EUR)
    df_hist['system_eur_kwh'] = df_hist['cell_usd_kwh'] * 2.5 * 0.92
    df_hist = df_hist[df_hist['Year'] >= 2015].sort_values('Year').dropna()
    yr_last  = int(df_hist['Year'].max())
    val_last = df_hist.loc[df_hist['Year'] == yr_last, 'system_eur_kwh'].values[0]
    print(f"OWID geladen: {len(df_hist)} Jahre ({int(df_hist['Year'].min())}–{yr_last})")
    print(f"  System-CAPEX {yr_last}: ~{val_last:.0f} EUR/kWh")
    print(f"  Quelle: {OWID_URL}")
except Exception as e:
    print(f"⚠️  OWID-Download fehlgeschlagen: {e}")
    print("   Fallback: eingebettete BNEF-Referenzwerte (Systemkosten EUR/kWh)")
    df_hist = pd.DataFrame({'Year': [2015,2016,2017,2018,2019,2020,2021,2022,2023,2024],
                            'system_eur_kwh': [530,445,380,310,255,215,190,178,160,145]})

years_h = df_hist['Year'].values.astype(float)
capex_h = df_hist['system_eur_kwh'].values.astype(float)

# ════════════════════════════════════════════════════════════════════════════
# DATENQUELLE 2: NREL ATB 2024 — Projektionsszenarien
# CSV aus OEDI Data Lake (AWS S3, öffentlich, kein Auth)
# https://oedi-data-lake.s3.amazonaws.com/ATB/electricity/csv/2024/v3.0.0/ATBe.csv
# ════════════════════════════════════════════════════════════════════════════
ATB_URL = ('https://oedi-data-lake.s3.amazonaws.com'
           '/ATB/electricity/csv/2024/v3.0.0/ATBe.csv')

# ATB CAPEX-Werte für Utility-Scale Battery (4h), $/kW → $/kWh = ÷4 → EUR/kWh × 0.92
# Struktur: technology | techdetail | scenario | year | parameter | value | units
ATB_SCENARIOS = None
try:
    print(f"\nLade NREL ATB 2024: {ATB_URL}")
    resp2 = requests.get(ATB_URL, timeout=30)
    resp2.raise_for_status()
    df_atb = pd.read_csv(io.StringIO(resp2.text))
    print(f"  ATB geladen: {df_atb.shape[0]:,} Zeilen | Spalten: {list(df_atb.columns)}")

    # Spalten normalisieren
    df_atb.columns = [c.lower().strip() for c in df_atb.columns]

    # Battery Storage filtern
    tech_col = next((c for c in df_atb.columns if 'tech' in c and 'detail' not in c), 'technology')
    param_col = next((c for c in df_atb.columns if 'param' in c), 'parameter')
    scen_col  = next((c for c in df_atb.columns if 'scen' in c), 'scenario')
    val_col   = next((c for c in df_atb.columns if 'value' in c), 'value')
    year_col  = next((c for c in df_atb.columns if 'year' in c), 'year')

    mask_bat  = df_atb[tech_col].str.lower().str.contains('battery|bess|storage', na=False)
    mask_cap  = df_atb[param_col].str.lower().str.contains('capex|capital', na=False)
    df_bat    = df_atb[mask_bat & mask_cap].copy()
    print(f"  Battery+CAPEX Zeilen: {len(df_bat)}")
    print(f"  Technologien: {df_bat[tech_col].unique()[:5]}")
    print(f"  Szenarien: {df_bat[scen_col].unique()}")

    # Utility-Scale 4h System bevorzugen
    detail_col = next((c for c in df_atb.columns if 'detail' in c), None)
    if detail_col:
        mask_4h = df_bat[detail_col].str.lower().str.contains('4', na=False)
        df_bat  = df_bat[mask_4h] if mask_4h.any() else df_bat

    # $/kW → EUR/kWh: CAPEX[$/kW] / 4h × 0.92 USD/EUR
    unit_col = next((c for c in df_bat.columns if 'unit' in c), None)
    df_bat['capex_eur_kwh'] = pd.to_numeric(df_bat[val_col], errors='coerce') / 4 * 0.92

    # Szenarien extrahieren
    ATB_SCENARIOS = {}
    for scen in ['Conservative', 'Moderate', 'Advanced']:
        df_s = df_bat[df_bat[scen_col].str.lower() == scen.lower()]
        if df_s.empty:
            # Partielle Übereinstimmung
            df_s = df_bat[df_bat[scen_col].str.lower().str.contains(scen.lower()[:4], na=False)]
        if not df_s.empty:
            ts = (df_s.groupby(year_col)['capex_eur_kwh'].mean()
                      .reset_index().sort_values(year_col))
            ATB_SCENARIOS[scen] = ts
            yr0, c0 = int(ts[year_col].iloc[0]), ts['capex_eur_kwh'].iloc[0]
            yr1, c1 = int(ts[year_col].iloc[-1]), ts['capex_eur_kwh'].iloc[-1]
            print(f"  Szenario {scen:12s}: {yr0}={c0:.0f} EUR/kWh → {yr1}={c1:.0f} EUR/kWh")
    print(f"  Quelle: {ATB_URL}")
except Exception as e:
    print(f"⚠️  NREL ATB Download fehlgeschlagen: {e}")
    print("   Fallback: NREL ATB Referenzwerte (embedded, 2022 Basisjahr, 4h System)")
    # Referenzwerte aus ATB 2024 Dokumentation (USD/kW ÷ 4 × 0.92)
    base_2022 = 185.0   # EUR/kWh (Utility 4h, 2022)
    proj_yrs  = list(range(2022, 2051))
    ATB_SCENARIOS = {
        'Conservative': pd.DataFrame({'year': proj_yrs,
            'capex_eur_kwh': [base_2022 * (1-0.014)**i for i in range(len(proj_yrs))]}),
        'Moderate':     pd.DataFrame({'year': proj_yrs,
            'capex_eur_kwh': [base_2022 * (1-0.029)**i for i in range(len(proj_yrs))]}),
        'Advanced':     pd.DataFrame({'year': proj_yrs,
            'capex_eur_kwh': [base_2022 * (1-0.040)**i for i in range(len(proj_yrs))]}),
    }
    print("   Raten: Conservative -1.4%/J | Moderate -2.9%/J | Advanced -4.0%/J")

# ════════════════════════════════════════════════════════════════════════════
# REGRESSION: Exponentielle Kurve auf historische OWID-Daten
# Proxy für Wright's Law auf der Zeitachse
# ════════════════════════════════════════════════════════════════════════════
def exp_decay(t, a, b):
    return a * np.exp(-b * t)

t_h = years_h - years_h[0]
try:
    popt, _ = curve_fit(exp_decay, t_h, capex_h, p0=[capex_h[0], 0.08], maxfev=5000)
    a_fit, b_fit = popt
    rate_pct = (1 - np.exp(-b_fit)) * 100
    print(f"\nRegression (Wright's Law Proxy):")
    print(f"  CAPEX(t) = {a_fit:.0f} × e^(−{b_fit:.4f}·t)  |  ~{rate_pct:.1f}%/Jahr historisch")
except Exception as e:
    print(f"Regression fehlgeschlagen: {e}"); b_fit = np.log(1/0.90); a_fit = capex_h[-1]

# Regressionskurve bis 2035 extrapolieren
base_yr    = int(years_h[-1])
base_capex = float(capex_h[-1])
proj_yrs_r = np.arange(base_yr, 2036)
proj_reg   = base_capex * np.exp(-b_fit * (proj_yrs_r - base_yr))

# ════════════════════════════════════════════════════════════════════════════
# CHART
# ════════════════════════════════════════════════════════════════════════════
SCEN_STYLE = {
    'Conservative': ('#FFA726', '--'),
    'Moderate':     ('#66BB6A', '--'),
    'Advanced':     ('#EF5350', ':'),
}

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.tick_params(colors='#bbbbbb')
for sp in ax.spines.values(): sp.set_edgecolor('#333')

# Historische Punkte (OWID)
ax.scatter(years_h, capex_h, color='white', s=45, zorder=6,
           label=f'Historisch — OWID/BNEF (Systemkosten)', alpha=0.9)
ax.plot(years_h, capex_h, color='white', lw=1.3, alpha=0.35)

# Regressionskurve
ax.plot(proj_yrs_r, proj_reg, color='#42A5F5', lw=2.2, ls='-',
        label=f'Regression histor. Daten (~{rate_pct:.0f}%/J)', alpha=0.9)

# NREL ATB Szenarien
if ATB_SCENARIOS:
    yr_col = list(ATB_SCENARIOS.values())[0].columns[0]
    for scen, df_s in ATB_SCENARIOS.items():
        col, ls = SCEN_STYLE.get(scen, ('#888888', ':'))
        mask = df_s[yr_col] >= base_yr
        ax.plot(df_s.loc[mask, yr_col], df_s.loc[mask, 'capex_eur_kwh'],
                color=col, lw=2, ls=ls, label=f'NREL ATB {scen}', alpha=0.9)

# Referenzlinien
ax.axvline(base_yr, color='white', lw=0.7, ls=':', alpha=0.3)
ax.axhline(250, color='#EF5350', lw=1.1, ls=':', alpha=0.6,
           label='Trigger-CAPEX 250 EUR/kWh (NB04)')
ax.axhline(150, color='#AB47BC', lw=1.0, ls=':', alpha=0.5,
           label='Langfrist-Ziel 150 EUR/kWh')
ax.axvspan(base_yr, 2035, alpha=0.04, color='white')
ax.text(base_yr + 0.3, max(capex_h) * 1.04, 'Projektion →', color='#666', fontsize=8)

ax.set_xlabel('Jahr', color='#aaa', fontsize=11)
ax.set_ylabel('System-CAPEX [EUR/kWh]\n'
              '(Zellen + WR + Installation, 4h-System, 2024-Preise)', color='#aaa', fontsize=10)
ax.set_title('Li-Ion Batteriespeicher — CAPEX-Entwicklung & Projektion\n'
             'Historisch: OWID/BNEF  |  Szenarien: NREL ATB 2024 (CSV)',
             color='white', fontweight='bold')
ax.legend(fontsize=8.5, framealpha=0.3, facecolor='#111', labelcolor='white',
          loc='upper right')
ax.grid(True, alpha=0.12)
ax.set_xlim(2014.5, 2035.5)

p = os.path.join(CHARTS_DIR, 'kuer_nb11_capex_lernkurven.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f'\nChart gespeichert: {p}')

**Befund:** Die historische Regression zeigt einen Rückgang von ~8–12 %/Jahr.
Das NREL ATB Moderate-Szenario (−2.9 %/Jahr) entspricht dem langfristigen Konsens;
das Advanced-Szenario (−4.0 %/Jahr) spiegelt aktuelle Dynamiken wider —
nach einem ~40 %-Rückgang 2024 liegen globale Projektkosten bereits bei ~125 USD/kWh
(Ember Energy, Januar 2026).

Der **Trigger-CAPEX von 250 EUR/kWh** (NB04) wird je nach Szenario
zwischen 2025 (Advanced) und 2028 (Conservative) unterschritten.

> **Datenquellen:**
> - [OWID — Li-Ion Battery Prices](https://ourworldindata.org/grapher/price-of-lithium-ion-battery-cells) (BNEF/Ziegler & Trancik, freier Download)
> - [NREL ATB 2024 — Utility-Scale Battery](https://atb.nrel.gov/electricity/2024/utility-scale_battery_storage) (CSV via AWS S3, freier Download)
> - **BloombergNEF Battery Price Survey** — umfassendste Branchendatenbank (Bezahlschranke; für geschäftliche Due-Diligence empfehlenswert)


---
## 4. Segmenteignung für Grid-Arbitrage CH

Eignungsscores 1–5 (5 = ideal) für die vier Projektsegmente.


In [ ]:
# ── Eignungsmatrix (Heatmap) ──────────────────────────────────────────────────
techs    = ['LFP', 'NMC', 'Redox-Flow', 'NaS']
segs     = ['Privat\n10 kWh', 'Gewerbe\n100 kWh', 'Industrie\n1 MWh', 'Utility\n10 MWh']
eignung  = np.array([
    [5, 4, 3, 2],   # LFP
    [3, 3, 2, 1],   # NMC
    [1, 2, 4, 5],   # Redox-Flow
    [1, 2, 3, 4],   # NaS
])
sterne = ['','★','★★','★★★','★★★★','★★★★★']

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
cmap = mcolors.LinearSegmentedColormap.from_list('', ['#1a1a1a','#1a3d2b','#5DCAA5'])
im = ax.imshow(eignung, cmap=cmap, vmin=1, vmax=5, aspect='auto')
ax.set_xticks(range(len(segs)));  ax.set_xticklabels(segs, color='#ccc', fontsize=10)
ax.set_yticks(range(len(techs))); ax.set_yticklabels(techs, color='#ccc', fontsize=10)
for i in range(len(techs)):
    for j in range(len(segs)):
        ax.text(j, i, sterne[eignung[i,j]], ha='center', va='center', color='white', fontsize=13)
ax.set_title('Technologie-Segmenteignung — Grid-Arbitrage CH', color='white', fontweight='bold')
cb = plt.colorbar(im, ax=ax, label='Eignung (1=gering, 5=ideal)', shrink=0.85)
cb.ax.tick_params(colors='#aaa'); cb.ax.yaxis.label.set_color('#aaa')
p = os.path.join(CHARTS_DIR, 'kuer_nb11_eignungsmatrix.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")

**Kernaussagen:**
- **LFP** ist die dominierende Technologie für Privat und Gewerbe — beste Sicherheit, tiefster CAPEX-Trend, marktreif.
- **NMC** verliert im stationären Bereich gegenüber LFP (Brandrisiko, kein CAPEX-Vorteil). Bleibt relevant für EV.
- **Redox-Flow** ist die strategisch interessanteste Alternative für Utility ≥ 10 MWh:
  nahezu unbegrenzte Zyklenlebensdauer, entkoppelbare Leistung/Energie, kaum Degradation.
- **NaS** benötigt 300–350 °C Betriebstemperatur — für dezentrale CH-Anlagen ungeeignet.


---
## 5. Fazit

**LFP ist die Referenztechnologie für alle Projektsegmente** — die in NB04 verwendeten
Parameter (RTE 92 %, CAPEX 150–400 EUR/kWh) sind korrekt und zeitgemäss.

Für Utility-Grossspeicher > 10 MWh ist **Redox-Flow** eine strategisch valide Alternative
mit deutlich längerer Lebensdauer (> 20 Jahre, > 20 000 Zyklen) und ohne Brandrisiko.
Der höhere CAPEX (400–800 EUR/kWh) amortisiert sich bei hohen Zyklenanzahlen.

> Für nicht-elektrochemische Alternativen (Pumpspeicher, CAES, Power-to-X)  
> → [NB12 Alternative Speicher](12_Alternative_Speicher.ipynb)


---

In [ ]:
# ── Abschlusskontrolle ───────────────────────────────────────────────────────
charts_out = [f for f in os.listdir(CHARTS_DIR) if f.startswith('kuer_nb11_')]
print("=== Abschlusskontrolle NB11 (Kür) ===")
for f in sorted(charts_out):
    print(f"  ✅  {f}")
print(f"  {len(charts_out)} Chart(s) erzeugt")
print("=== Ende ===")


---
| [↑ Projektübersicht](00_Project_Overview.ipynb) | [← NB10 Dispatch](10_Dispatch_Optimierung.ipynb) | [→ NB12 Alternative Speicher](12_Alternative_Speicher.ipynb) |
|:---|:---:|---:|
